# $\zeta(2)$: historical constructions and new best trajectory

Most CMF examples use depths 10, 20, ..., 3000 (300 points per curve). Giraudin uses depths 1, 2, 3, 5, 10, and the Dvornicichג€“Viola staircase uses
$m=10,20,\ldots,900$, plotted with $N=m$.

Each example prints **$\delta$ at its stated final depth**; two figures at the
end plot $\delta_N$ against $N$ and against $D_N$ for every trajectory, where
$$D_N=-\log_{10}\left|\zeta(2)-\frac{p_N}{q_N}\right|,
\qquad \delta_N=\frac{D_N}{\log_{10}q_N}-1.$$
Here $p_N/q_N$ is reduced. Accuracy digits measure the absolute approximation error,
not the working precision or the matrix-walk depth.

Run the cells in order (Python 3.11+). The historical CMF encodings follow the supplied
`Zeta(2)_p (1).ipynb`; the new best trajectory is the later search candidate
$(10,9,11,-11;2,1,3)$. The newly verified recoveries use exact source indices through 1000.
Published bounds and numerical $\delta_N$ are distinct: finite-depth computations
do not prove an irrationality-measure bound or the identification with a historical construction.

The explicit Apéry and Zudilin (2002) recoveries below sample source indices
10,20,...,1000. Their CMF walk length is N-2; earlier sections retain their
stated sampling conventions. Only the changed recovery cells were rerun.

Dauguet–Zudilin (2014) also uses source indices through 1000, equal to its walk depths.


In [ ]:
# @title Install dependencies (run once per environment)
%pip -q install ramanujantools==0.0.8 sympy==1.14.0 mpmath==1.3.0 matplotlib==3.11.2

## Setup

Each example constructs a CMF, walks a trajectory with ramanujantools, reads
$p_N/q_N$ off the first column $(s_0,s_1,\ldots)$ of the walk, and computes
$\delta_N$. The walks and the approximants are exact.

In [2]:
from cmfs import apery_seeds, zudilin_zeta23_seeds, zudilin_zeta23_readout
import sys

import matplotlib.pyplot as plt
import sympy as sp
from flint import arb, ctx
from ramanujantools.cmf.pfq import pFq

sys.set_int_max_str_digits(0)
with ctx.workdps(160_000):  # q_N reaches about 76,500 digits at depth 3000 (Hata)
    ZETA2 = arb(2).zeta()
DEPTHS = list(range(10, 3001, 10))
x0, x1, x2, x3, y0, y1, y2 = sp.symbols("x0 x1 x2 x3 y0 y1 y2")
results = {}
accuracy = {}


def delta(approximant):
    """delta with |־¶(2) - p/q| = q^-(1 + delta), for the reduced fraction p/q."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2 * q.bit_length() + 64):  # enough bits to resolve |־¶(2) - p/q|
        error = abs(ZETA2 - arb(p) / q)
        assert error > 0
        return float((-1 - error.log() / arb(q).log()).mid())

def correct_digits(approximant):
    """D_N = -log10 |־¶(2) - p/q|, for the reduced fraction p/q."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2 * q.bit_length() + 64):
        error = abs(ZETA2 - arb(p) / q)
        assert error > 0
        return float((-error.log() / arb(10).log()).mid())

## 1. Apéry (1978 announcement; 1979 publication): exact source sequences

[Original paper, pp. 12–13](https://www.numdam.org/article/AST_1979__61__11_0.pdf).
Both sequences satisfy
$$(N+1)^2u_{N+1}-(11N^2+11N+3)u_N-N^2u_{N-1}=0,$$
with $(a_0,a_1)=(1,3)$ and $(b_0,b_1)=(0,5)$;
the approximants are $b_N/a_N$. The recurrence normalization is also recovered at
$\alpha=0$ in [Zudilin (2003)](https://arxiv.org/abs/math/0307058).
Apéry's asymptotic measure is
$\mu_0=2\log\lambda/(\log\lambda-2)\approx 11.85078$, $\lambda=(11+5\sqrt5)/2$.

**Full CMF:** ${}_3F_2(-N,-N,N+1;1,1;1)$, rank 2, 5 independent parameter axes.
**Start:** $(-2,-2,3;1,1)$ (source index 2); **direction:** $(-1,-1,1;0,0)$.
With $W$ the walk of length $N-2$, the exact readout is $(PW)_0/(QW)_0$, where
$$P=(125/4,79/2),\quad Q=(19,24).$$
The denominator is the terminating binomial sum
$a_N=\sum_{k=0}^N\binom Nk^2\binom{N+k}k^{1}$.
The full-field trajectory satisfies the displayed recurrence as a symbolic identity;
its first two source values fix the numerator and denominator rows.

`local_checks/verify_apery_zudilin_zeta23.py` checks that identity, the binomial sums at small
indices, exact source fractions at every sampled index through 1000, and all axis
pairs at two off-trajectory rational points. Two Arb precisions give
$\delta_{1000}=0.102022424091$. This measured exponent is distinct from the published bound.


In [ ]:
cmf = pFq(3, 2, 1)
axes = sp.symbols("x0:3") + sp.symbols("y0:2")
start = dict(zip(axes, [-2, -2] + [3]*1 + [1]*2))
trajectory = dict(zip(axes, [-1, -1] + [1]*1 + [0]*2))
depths = list(range(10, 1001, 10))  # source indices
P, Q = apery_seeds(2)
walks = cmf.walk(trajectory, [N - 2 for N in depths], start)
approximants = [(P*W)[0]/(Q*W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Apéry (1979), exact source index"] = (depths, deltas)
accuracy["Apéry (1979), exact source index"] = [correct_digits(r) for r in approximants]
print(f"delta at source index 1000 (998 steps): {deltas[-1]:.12f}")


delta at source index 1000 (998 steps): 0.102022424091


## 2. Zudilin (2002): simultaneous zeta(2), zeta(3) approximations

[Original paper, Section 3, equation (16) and Theorem 4](https://arxiv.org/abs/math/0206178).
This is a distinct third-order construction. Its proven convergence rate is
$$\lim_{N\to\infty}\frac1N\log|\zeta(s)-p_N^{(s)}/q_N|
=\log|\mu_2|-\log|\mu_3|,$$
where $\mu_{1,2}=0.07260980\ldots\pm0.25981363\ldots i$,
$\mu_3=219.85478039\ldots$ solve $\mu^3-220\mu^2+32\mu-16=0$.
No irrationality-measure bound is claimed for this recovery.

**Full CMF:** rank-3 ${}_4F_3(1)$ with all seven independent axes. Define
$$F_N(z)={}_4F_3(N+1,N+1,N+1,N+1;2N+2,2N+2,2N+2;z),
\quad C_N=\frac{N!^6}{(2N+1)!^3},\quad\theta=z\frac{d}{dz}.$$
The paper's auxiliary series are exactly $-C_NF_N(1)$ and
$C_N[(\theta+N+1)F_N]_{z=1}$ after the shift $k=m+N+1$.
The state is $(F,\theta F,\theta^2F)$.
**Start:** $(3,3,3,3;6,6,6)$, source index 2;
**direction:** $(1,1,1,1;2,2,2)$; **walk length:** $N-2$.

Exact seed rows in `cmfs.py` come from partial fractions of
$$\frac{(5!)^3}{(2!)^4}\frac{(m+1)(m+2)}{(m+3)^3(m+4)^3(m+5)^3}.$$
Write $F_N(1)=A_0+A_2\zeta(2)+A_3\zeta(3)$ and
$[(\theta+N+1)F_N]_{z=1}=B_0+B_2\zeta(2)+B_3\zeta(3)$.
With $D=A_3B_2-B_3A_2$, the two exact readouts are
$$\frac{p_N^{(2)}}{q_N}=\frac{B_3A_0-A_3B_0}{D},\qquad
\frac{p_N^{(3)}}{q_N}=\frac{A_2B_0-B_2A_0}{D}.$$
Multiplication by $C_N^2$ reproduces the source normalization:
$q=(1,14,978,\ldots)$, $p^{(2)}=(0,23,6435/4,\ldots)$,
$p^{(3)}=(0,17,9405/8,\ldots)$.

The verification script proves recurrence (16) symbolically for the normalized
two-by-two minors of the full-field walk, and checks the normalized forms at source indices 2–9,
the recurrence (16) at every sampled index through 1000, and both precision runs.
The $\zeta(2)$ readout gives $\delta_{1000}=-0.038603691054$; this is a negative finite-depth approximation exponent.


In [ ]:
cmf = pFq(4, 3, 1)
axes = sp.symbols("x0:4") + sp.symbols("y0:3")
start = dict(zip(axes, [3]*4 + [6]*3))
trajectory = dict(zip(axes, [1]*4 + [2]*3))
depths = list(range(10, 1001, 10))  # source indices
seeds = zudilin_zeta23_seeds()
walks = cmf.walk(trajectory, [N - 2 for N in depths], start)
approximants = [zudilin_zeta23_readout(W, N, seeds, 2) for N, W in zip(depths, walks)]
deltas = [delta(r) for r in approximants]
results["Zudilin (2002), simultaneous"] = (depths, deltas)
accuracy["Zudilin (2002), simultaneous"] = [correct_digits(r) for r in approximants]
print(f"delta at source index 1000 (998 steps): {deltas[-1]:.12f}")


delta at source index 1000 (998 steps): -0.038603691054


## 3. Dauguet–Zudilin (2014): simultaneous construction

S. Dauguet and W. Zudilin, [*On simultaneous diophantine approximations to
$\zeta(2)$ and $\zeta(3)$*, Section 3, equations (25)–(30)](https://arxiv.org/abs/1401.5322).
Their Theorem 1 gives the specialized simultaneous bound
$s_{\tau_0}(\zeta(2),\zeta(3))\le s_0$, with
$\tau_0=0.899668635\ldots$ and $s_0=6.770732145\ldots$.
This exponent imposes specific denominator divisibility conditions;
**$s_0$ is not an upper bound for the ordinary irrationality measure $\mu$.**
The source proves growth $\log|q_N|/N\to27.86755317\ldots$ and
$\limsup\log|q_N\zeta(s)-p_N^{(s)}|/N=-19.10095491\ldots$ for both readouts.

**Full CMF:** rank-3 ${}_4F_3(1)$, all seven independent parameter axes.
**Start:** $(1,1,1,0;1,1,1)$;
**direction:** $(10,9,8,-5;1,2,3)$.
The walk depth $N$ is the source index, with no offset.
If $(w_0,w_1,w_2)^T$ is the first column of the direct walk, the two readouts are
$$\frac{p_N^{(2)}}{q_N}=\frac{2w_1-w_2}{2w_0},\qquad
\frac{p_N^{(3)}}{q_N}=\frac{w_2}{2w_0}.$$
The common normalization is
$$f_N=\frac{(-1)^N(9N)!(10N)!}{N!(2N)!(3N)!(5N)!(8N)!},
\quad(q_N,p_N^{(2)},p_N^{(3)})=f_N(w_0,w_1-w_2/2,w_2/2).$$
In particular the denominator matches the terminating ${}_4F_3$ in (27).
The paper proves that the two constructions share a third-order recurrence.

`local_checks/verify_dauguet_zudilin2014.py` independently expands the defining rational
functions (5) and (15), specialized by (25) and (28), into partial fractions.
It checks both normalized numerator sequences, the shared denominator and the
finite sum (27) at indices 0–10, all axis pairs at two off-trajectory points,
and the depth-1000 errors at two Arb precisions. The source's prime-divisibility
argument is referenced, not re-proved by this computation.

**Source discrepancy:** the displayed initial-value line on p. 15 prints
$q_1=12307565655$ and $q_2=5669931265166541788415$.
Both defining constructions and finite sum (27) instead give
$q_1=12307605345$ and $q_2=5669931265185834471825$.
We use these formula-derived values; the printed numerator values agree.
The overall sign is normalized to (27), which cancels in the rational readouts.
For $\zeta(2)$, the verified finite-depth exponent is $\delta_{1000}=0.060969252271$.


In [ ]:
cmf = pFq(4, 3, 1)
axes = sp.symbols("x0:4") + sp.symbols("y0:3")
start = dict(zip(axes, [1, 1, 1, 0, 1, 1, 1]))
trajectory = dict(zip(axes, [10, 9, 8, -5, 1, 2, 3]))
depths = list(range(10, 1001, 10))
walks = cmf.walk(trajectory, depths, start)
approximants = [(2*W[1,0]-W[2,0])/(2*W[0,0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["Dauguet–Zudilin (2014)"] = (depths, deltas)
accuracy["Dauguet–Zudilin (2014)"] = [correct_digits(r) for r in approximants]
print(f"delta at source index 1000: {deltas[-1]:.12f}")


delta at source index 1000: 0.060969252271


## 4. Beukers (1979)

F. Beukers, *A note on the irrationality of $\zeta(2)$ and $\zeta(3)$*,
Bull. London Math. Soc. **11** (1979), 268ג€“272.
[Paper](https://doi.org/10.1112/blms/11.3.268).

$$I_N=\int_0^1\!\int_0^1
\frac{[x(1-x)y(1-y)]^N}{(1-xy)^{N+1}}\,dx\,dy.$$

**CMF:** ${}_3F_2(1)$ ֲ· **Start:** $(3,3,3;6,6)$ ֲ· **Direction:** $(1,1,1;2,2)$.
We use the inverse-transpose walk. The extraction is $p_N/q_N=(533s_0+125s_1)/(324s_0+76s_1)$. This start matches the supplied notebook and
lies two steps beyond $(1,1,1;2,2)$.

In [3]:
cmf = pFq(3, 2, 1).dual()  # inverse-transpose matrices
start = {x0: 3, x1: 3, x2: 3, y0: 6, y1: 6}
trajectory = {x0: 1, x1: 1, x2: 1, y0: 2, y1: 2}
walks = cmf.walk(trajectory, DEPTHS, start)
approximants = [(533 * W[0, 0] + 125 * W[1, 0]) / (324 * W[0, 0] + 76 * W[1, 0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["Beukers (1979)"] = (DEPTHS, deltas)
accuracy["Beukers (1979)"] = [correct_digits(r) for r in approximants]
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.093976576


## 5. Dvornicichג€“Viola (1987; proceedings 1990)

R. Dvornicich and C. Viola, *Some remarks on Beukers' integrals*,
Colloq. Math. Soc. Jֳ¡nos Bolyai **51**, 637ג€“657. The construction is restated in
[Rhinג€“Viola (1993), p. 100](https://www.numdam.org/article/AIF_1993__43_1_85_0.pdf): with $\psi=x(1-x)y(1-y)/(1-xy)$,
$$\int_0^1\!\int_0^1\psi^{n_0}(1-11\psi)^{n_1}(1-12\psi)^{n_2}\,\frac{dx\,dy}{1-xy},
\qquad n_j=2\left[\tfrac{a_j}{2}n\right],$$
with $(a_0,a_1,a_2)=(0.885856\ldots,0.066342\ldots,0.047802\ldots)$; the exponents are even,
so $(11\psi-1)^{n_1}=(1-11\psi)^{n_1}$. Taking $n_2=2m$, the slopes
$a_0/a_2=18.53172$ and $a_1/a_2=1.38784$ give the path below.

**CMF:** $D_2(e_0,e_1,e_2)$, the moment CMF of
$S(e_0,e_1,e_2)=\iint\psi^{e_0}(1-11\psi)^{e_1}(1-12\psi)^{e_2}\,dx\,dy/(1-xy)$
(`dvornicich_viola_zeta2`, rank 6) ֲ· **Start:** $(0,0,0)$ ֲ·
**Path:** the staircase $(2\lfloor18.53172m\rfloor,2\lfloor1.38784m\rfloor,2m)$, $m=10,20,\ldots,900$ ֲ·
$S=C+Z\zeta(2)$, $p_m/q_m=-C_0/Z_0$.

The moments $K_\ell=S(\ell,0,0)$ are Beukers' integrals, $K_\ell=(-1)^\ell(a_\ell\zeta(2)-b_\ell)$
with Apֳ©ry's numbers $a_\ell,b_\ell$, and
$(\ell+1)^2K_{\ell+1}+(11\ell^2+11\ell+3)K_\ell-\ell^2K_{\ell-1}=0$ for $\ell\ge1$.
This gives one recurrence in $e_0$ at every $(e_1,e_2)$; $M_{e_0}$ is its companion matrix,
$M_{e_1}=I-11M_{e_0}$ and $M_{e_2}=I-12M_{e_0}$. The state is
$(S(e_0,e_1,e_2),\ldots,S(e_0+5,e_1,e_2))$. The walk steps one axis at a time through the
staircase points; the steps between two samples are multiplied as a balanced tree before they act
on the state, which has tens of thousands of digits at the end. The finite-depth exponent below
is a numerical quantity, not the published bound $\mu(\zeta(2))\le10.0298$.

In [4]:
from flint import fmpq, fmpq_mat

from cmfs import axis_steps, dvornicich_viola_zeta2

e0, e1, e2 = sp.symbols("e0 e1 e2")
M = axis_steps(dvornicich_viola_zeta2())  # M(x, point): the axis matrix M_x at an integer point
# Apery's numbers for zeta(2): K_l = (-1)^l (a_l zeta(2) - b_l).
a, b = [1, 3], [fmpq(0), fmpq(5)]
for l in range(1, 2000):  # enough terms for the readout check below
    a.append(((11 * l**2 + 11 * l + 3) * a[l] + l**2 * a[l - 1]) // (l + 1) ** 2)
    b.append(((11 * l**2 + 11 * l + 3) * b[l] + l**2 * b[l - 1]) / (l + 1) ** 2)
# The rows C and Z of the start state (K_0, ..., K_5), with K_l = C_l + Z_l zeta(2).
dv_start = fmpq_mat([[(-1) ** (l + 1) * b[l] for l in range(6)], [(-1) ** l * a[l] for l in range(6)]])


def staircase(m):
    """The point (2 floor(18.53172 m), 2 floor(1.38784 m), 2m) of the Dvornicich--Viola path."""
    return {e0: 2 * (1853172 * m // 10**5), e1: 2 * (138784 * m // 10**5), e2: 2 * m}

def product(matrices):
    """The ordered product of `matrices`, multiplied pairwise as a balanced tree."""
    while len(matrices) > 1:
        matrices = [matrices[i] * matrices[i + 1] if i + 1 < len(matrices) else matrices[i]
                    for i in range(0, len(matrices), 2)]
    return matrices[0]

dv_depths = list(range(10, 901, 10))
state, point, previous = dv_start, staircase(0), 0
approximants = []
for m in dv_depths:
    steps = []
    for j in range(previous + 1, m + 1):
        target = staircase(j)
        for x in (e0, e1, e2):
            while point[x] < target[x]:
                steps.append(M(x, point))
                point[x] += 1
    state, previous = state * product(steps), m
    approximants.append(-state[0, 0] / state[1, 0])
deltas = [delta(r) for r in approximants]
results["Dvornicichג€“Viola (1990)"] = (dv_depths, deltas)
accuracy["Dvornicichג€“Viola (1990)"] = [correct_digits(r) for r in approximants]
print(f"delta at m = 900: {deltas[-1]:.9f}")

delta at m = 900: 0.109966371


**Readout check.** If $(1-11t)^{e_1}(1-12t)^{e_2}=\sum_jr_jt^j$, then
$S(e_0,e_1,e_2)=\sum_jr_jK_{e_0+j}$. The next cell computes these sums directly from Apֳ©ry's numbers
and checks that they equal the first column of the walked state, exactly, at the first 30 staircase
points, independently of the CMF matrices. It also checks the start values $K_0=\zeta(2)$ and
$K_1=5-3\zeta(2)$ against numerical quadrature of the double integral.

In [5]:
import mpmath

t = sp.Symbol("t")
state, point = dv_start, staircase(0)
for m in range(1, 31):
    target = staircase(m)
    for x in (e0, e1, e2):
        while point[x] < target[x]:
            state = state * M(x, point)
            point[x] += 1
    r = sp.Poly((1 - 11 * t) ** target[e1] * (1 - 12 * t) ** target[e2], t).all_coeffs()[::-1]
    l = target[e0]
    C = sum(int(r_j) * (-1) ** (l + j + 1) * b[l + j] for j, r_j in enumerate(r))
    Z = sum(fmpq(int(r_j) * (-1) ** (l + j) * a[l + j]) for j, r_j in enumerate(r))
    assert state[0, 0] == C and state[1, 0] == Z, m
print("walk = direct moment sums at m = 1, ..., 30")

mpmath.mp.dps = 20
psi = lambda x, y: x * (1 - x) * y * (1 - y) / (1 - x * y)
for l in range(2):
    K = mpmath.quad(lambda x, y: psi(x, y) ** l / (1 - x * y), [0, 1], [0, 1])
    print(f"K_{l}: quadrature {mpmath.nstr(K, 15)}, Apery {mpmath.nstr((-1) ** l * (a[l] * mpmath.zeta(2) - int(b[l].p)), 15)}")

walk = direct moment sums at m = 1, ..., 30


K_0: quadrature 1.64493406684823, Apery 1.64493406684823
K_1: quadrature 0.0651977994553207, Apery 0.0651977994553207


## 6. Hata (1995)

M. Hata, *A note on Beukers' integral*, J. Austral. Math. Soc. (Series A)
**58** (1995), 143ג€“153. [Paper](https://doi.org/10.1017/S1446788700038192).
The bound $\mu(\zeta(2))<5.687$ appears in the **Addendum**.

**CMF:** ${}_3F_2(1)$ ֲ· **Start:** $(3,3,3;6,6)$ ֲ· **Direction:** $(12,15,14;30,28)$.
We use the inverse-transpose walk. The extraction is $p_N/q_N=(533s_0+125s_1)/(324s_0+76s_1)$.

In [6]:
cmf = pFq(3, 2, 1).dual()  # inverse-transpose matrices
start = {x0: 3, x1: 3, x2: 3, y0: 6, y1: 6}
trajectory = {x0: 12, x1: 15, x2: 14, y0: 30, y1: 28}
walks = cmf.walk(trajectory, DEPTHS, start)
approximants = [(533 * W[0, 0] + 125 * W[1, 0]) / (324 * W[0, 0] + 76 * W[1, 0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["Hata (1995)"] = (DEPTHS, deltas)
accuracy["Hata (1995)"] = [correct_digits(r) for r in approximants]
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.216929455


## 7. Rhinג€“Viola (1996)

G. Rhin and C. Viola, *On a permutation group related to $\zeta(2)$*,
Acta Arith. **77** (1996), 23ג€“56.
[Paper](https://matwbn.icm.edu.pl/ksiazki/aa/aa77/aa7713.pdf).

**CMF:** ${}_3F_2(1)$ ֲ· **Start:** $(3,3,3;6,6)$ ֲ· **Direction:** $(12,13,14;24,28)$.
We use the inverse-transpose walk. The extraction is $p_N/q_N=(533s_0+125s_1)/(324s_0+76s_1)$.

In [7]:
cmf = pFq(3, 2, 1).dual()  # inverse-transpose matrices
start = {x0: 3, x1: 3, x2: 3, y0: 6, y1: 6}
trajectory = {x0: 12, x1: 13, x2: 14, y0: 24, y1: 28}
walks = cmf.walk(trajectory, DEPTHS, start)
approximants = [(533 * W[0, 0] + 125 * W[1, 0]) / (324 * W[0, 0] + 76 * W[1, 0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["Rhinג€“Viola (1996)"] = (DEPTHS, deltas)
accuracy["Rhinג€“Viola (1996)"] = [correct_digits(r) for r in approximants]
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.225083194


## 8. Zudilin (2014; preprint 2013)

W. Zudilin, *Two hypergeometric tales and a new irrationality measure of $\zeta(2)$*,
Ann. Math. Quֳ©bec **38** (2014), 101ג€“117.
[Paper](https://arxiv.org/abs/1310.1526).

**CMF:** ${}_4F_3(1)$ ֲ· **Start:** $(1,1,1,0;1,1,1)$ ֲ· **Direction:** $(8,7,6,-6;1,2,3)$.
Here we use the **direct** walk and the extraction
$p_N/q_N=(2s_1-s_2)/(2s_0)$.

In [8]:
cmf = pFq(4, 3, 1)
start = {x0: 1, x1: 1, x2: 1, x3: 0, y0: 1, y1: 1, y2: 1}
trajectory = {x0: 8, x1: 7, x2: 6, x3: -6, y0: 1, y1: 2, y2: 3}
walks = cmf.walk(trajectory, DEPTHS, start)
approximants = [(2 * W[1, 0] - W[2, 0]) / (2 * W[0, 0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["Zudilin (2014)"] = (DEPTHS, deltas)
accuracy["Zudilin (2014)"] = [correct_digits(r) for r in approximants]
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.243990507


## Giraudin (2026 preprint)

D. Niedbala Giraudin, [*A new upper bound for the irrationality exponent of\(\zeta(2)\)*](https://arxiv.org/abs/2610.02912).
The first Barnes construction uses
$(\alpha_1,\alpha_2,\alpha_3,\alpha_4;\beta_1,\beta_2,\beta_3,\beta_4)$
$=(146,123,100,169;0,23,46,292)$. Applying the same parameter
conversion and readout as for Zudilin above gives the direct ${}_4F_3(1)$
trajectory $x_0=(1,1,1,0;1,1,1)$,
$v=(169,146,123,-123;23,46,69)$, and
$p_N/q_N=(2s_1-s_2)/(2s_0)$. The paper's published bound
$\mu(\zeta(2))<5.0193784$ additionally uses its second construction
and arithmetic divisibility; the finite-depth CMF exponent below is a
numerical quantity, not that bound.


In [9]:
# Giraudin's first (Barnes / terminating 4F3) construction.
cmf = pFq(4, 3, 1)
start = {x0: 1, x1: 1, x2: 1, x3: 0, y0: 1, y1: 1, y2: 1}
trajectory = {x0: 169, x1: 146, x2: 123, x3: -123, y0: 23, y1: 46, y2: 69}
giraudin_depths = [1, 2, 3, 5, 10]
walks = cmf.walk(trajectory, giraudin_depths, start)
approximants = [(2 * W[1, 0] - W[2, 0]) / (2 * W[0, 0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["Giraudin (2026)"] = (giraudin_depths, deltas)
accuracy["Giraudin (2026)"] = [correct_digits(r) for r in approximants]
print(f"delta at depth {giraudin_depths[-1]}: {deltas[-1]:.9f}")

delta at depth 10: 0.252381576


## 9. New best trajectory

The search candidate $(10,9,11,-11;2,1,3)$. It replaces the trajectory
$(50,42,38,-42;4,12,18)$ recorded in the supplied `Zeta(2)_p (1).ipynb` (cell 27):
at matched accuracy digits the candidate gives $\delta_N\approx0.2785$, against
$\delta_N\approx0.2745$ for the earlier trajectory.

**CMF:** ${}_4F_3(1)$ ֲ· **Start:** $(1,1,1,0;1,1,1)$

**Direction:** $(10,9,11,-11;2,1,3)$.
We use the **direct** walk and the extraction $p_N/q_N=(2s_1-s_2)/(2s_0)$, as for
Zudilin's direction above. The result below is a numerical exponent for this trajectory;
the paper proves $\mu(\zeta(2))<4.58989684$ (and $\delta>0.27855953$) along this direction.


In [10]:
cmf = pFq(4, 3, 1)
start = {x0: 1, x1: 1, x2: 1, x3: 0, y0: 1, y1: 1, y2: 1}
trajectory = {x0: 10, x1: 9, x2: 11, x3: -11, y0: 2, y1: 1, y2: 3}
walks = cmf.walk(trajectory, DEPTHS, start)
approximants = [(2 * W[1, 0] - W[2, 0]) / (2 * W[0, 0]) for W in walks]
deltas = [delta(r) for r in approximants]
results["New best trajectory"] = (DEPTHS, deltas)
accuracy["New best trajectory"] = [correct_digits(r) for r in approximants]
print(f"delta at depth 3000: {deltas[-1]:.9f}")

delta at depth 3000: 0.278673338


The paper places the new direction about $12^\circ$ from Zudilin's
$(8,7,6,-6;1,2,3)$ (Euclidean angle between the direction vectors).

In [11]:
import math

zudilin = (8, 7, 6, -6, 1, 2, 3)
new = (10, 9, 11, -11, 2, 1, 3)
cosine = sum(a * b for a, b in zip(zudilin, new)) / (math.hypot(*zudilin) * math.hypot(*new))
print(f"angle between the directions: {math.degrees(math.acos(cosine)):.2f} degrees")

angle between the directions: 12.41 degrees


## All constructions

$\delta_N$ against the walk depth $N$.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(depths, values, label=name)
ax.set(xlabel="$N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## $\delta_N$ against accuracy digits

The same samples against $D_N$ instead of $N$, as in the paper's figure of
$\delta_N$ against accuracy digits.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(accuracy[name], values, label=name)
ax.set(xlabel="$D_N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## Hata (1990): no executed CMF calculation

The original construction (pp. 117ג€“120 of
[the paper](https://doi.org/10.1515/crll.1990.407.99)) inserts additional
polynomial factors $(1-2y)$, $(2-3y)$, and $(3-4y)$ into a Beukers-type
integral. Their multiplicities use floors of optimized real slopes.
The reconstruction therefore needs a field for those polynomial insertions
and its integer staircase path. A single direction in the existing
${}_3F_2(1)$ field has not been verified for this row.



## Rhinג€“Viola (1993): no executed CMF calculation

Section 6.2, fourth case, of
[the original paper](https://www.numdam.org/article/AIF_1993__43_1_85_0.pdf)
specifies three extra polynomial insertions. With
$B=xy(1-x)(1-y)$, define
$$Q=(2x-1)(2y-1)(2xy-1)(1-2x+xy)(1-2y+xy),$$
$$R=[(x-y)(1-2x+x^2y)(1-2y+xy^2)(1-x-xy)(1-y-xy)]^2,$$
$$S=(3x-2)(3y-2)(3xy-1)(1-3x+2xy)(1-3y+2xy).$$
The printed rational approximations to its optimized slopes suggest the
candidate subsequence
$$\int_{[0,1]^2}\frac{B^{28036N}Q^{1244N}R^{701N}S^{136N}}
{(1-xy)^{35002N+1}}\,dx\,dy.$$
The relation $28036+2(1244)+6(701)+2(136)=35002$ checks its normalization.
This identifies a concrete integral family to encode; it is **not** yet
a verified CMF or a certification of the published bound. In particular,
the printed slopes are approximate optimizers, not an asserted exact
reproduction of the historical extremum.


## Historical overview

Published upper bounds are listed for reference; they are not fitted to the computed
$\delta_N$. The corresponding comparison scale is $\delta_{\rm eq}=1/(\mu_0-1)$.

| Construction | Published upper bound | Notebook coverage |
|:--|--:|:--|
| Apéry (1978/1979) | $11.85078$ | Full CMF; source $N=1000$, $\delta=0.102022424091$ |
| Zudilin (2002), simultaneous | Convergence; no irrationality bound | Full CMF; source $N=1000$, $\delta=-0.038603691054$ |
| Dauguet–Zudilin (2014), simultaneous | $s_{\tau_0}\le6.770732145\ldots$ (not $\mu$) | Full CMF; $\delta_{1000}=0.060969252271$ |
| Beukers (1979) | $11.85078$ | Earlier integral encoding retained |
| Dvornicichג€“Viola (1987) | $10.0298$ | CMF above ($m=900$) |
| Hata (1990) | $7.5252$ | Integral discussed above; CMF not verified |
| Rhinג€“Viola (1993) | $7.398537$ | Integral discussed above; CMF not verified |
| Hata (1995), Addendum | $5.687$ | CMF above |
| Rhinג€“Viola (1996) | $5.441243$ | CMF above |
| Zudilin (2014) | $5.095412$ | CMF above |
| [Giraudin (2026 preprint)](https://arxiv.org/abs/2610.02912) | $5.0193784$ | first Barnes construction: CMF above ($N=10$) |
| This work | $4.58989684$ | New best trajectory above |

**Additional references.** The chronology and first four bounds appear in
[Rhinג€“Viola (1993), introduction, p. 86](https://www.numdam.org/article/AIF_1993__43_1_85_0.pdf).
The 1996 and 2014 bounds are also stated in
[Zudilin (2014), p. 1](https://arxiv.org/abs/1310.1526).

- R. Dvornicich and C. Viola, *Some remarks on Beukers' integrals*,
  Colloq. Math. Soc. Jֳ¡nos Bolyai **51**, 637ג€“657.
  [Author's publication record](https://arpi.unipi.it/handle/11568/10098).
  The year 1987 follows the historical attribution in Rhinג€“Viola.
- M. Hata, *Legendre type polynomials and irrationality measures*,
  J. Reine Angew. Math. **407** (1990), 99ג€“125.
  [Publication](https://doi.org/10.1515/crll.1990.407.99).
- G. Rhin and C. Viola, *On the irrationality measure of $\zeta(2)$*,
  Ann. Inst. Fourier **43** (1993), 85ג€“109.
  [Paper](https://www.numdam.org/article/AIF_1993__43_1_85_0.pdf).

The two entries without an executed CMF calculation have no computed $\delta_{3000}$ or plot here.
Their published bounds do not determine a CMF trajectory. This is a selection of
published records, not an exhaustive list of announced estimates.
